# Model Training

## **Setup**

In [ ]:
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from scipy.spatial.distance import cdist
import numpy as np
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as cx
import folium
# import os

# --- Configuration ---
pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")


In [ ]:
# Get Point Reyes boundaries
# ================================================
cpad = gpd.read_file("../Data/CPAD_Release_2025b/CPAD_2025b_Units/CPAD_2025b_Units.shp")
pt_reyes = cpad[cpad['UNIT_NAME'] == 'Point Reyes National Seashore'].copy()
pt_reyes_boundary = pt_reyes.dissolve()
# Save
# pt_reyes_boundary.to_file("point_reyes_boundary.geojson", driver='GeoJSON')

# Plot
# ================================================
# Reproject to Web Mercator (EPSG:3857)
pt_reyes_web_mercator = pt_reyes_boundary.to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 10))
pt_reyes_web_mercator.plot(ax=ax, alpha=1.0, facecolor='none', edgecolor='r', linewidth=2)
cx.add_basemap(ax, source=cx.providers.Esri.WorldTopoMap)
ax.set_axis_off()
plt.show()

# **Temporal Component - Weather conditions and mushroom emergence**

### Open-Meteo Data

Open-Meteo also has historical reanalysis data, but on a coarser 9km grid.

In [ ]:
reanalysis_om_df = pd.read_csv('../Data/weather_data_open-meteo/data/pt_reyes_weather_history_om.csv')
reanalysis_om_df.head()

reanalysis_om_df['date'] = pd.to_datetime(reanalysis_om_df['date'])
reanalysis_om_df['prcp_mm_7d_ma'] = reanalysis_om_df['prcp_mm'].rolling(window=7, min_periods=1).mean()

In [ ]:
import matplotlib.dates as mdates

fig, ax1 = plt.subplots(figsize=(16, 5))

ax1.bar(reanalysis_om_df['date'], reanalysis_om_df['prcp_mm'], color='blue', label='Precipitation (mm)', edgecolor='none')
ax1.set_ylabel('Precipitation (mm)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(reanalysis_om_df['date'], reanalysis_om_df['tmax_c'], color='red', linewidth=0.2, label='Tmax (°C)')
ax2.plot(reanalysis_om_df['date'], reanalysis_om_df['tmin_c'], color='teal', linewidth=0.2, label='Tmin (°C)')
ax2.set_ylabel('Temperature (°C)')
ax2.tick_params(axis='y')

# Optionally add a legend for temperature lines
lines, labels = ax2.get_legend_handles_labels()
ax2.legend(lines, labels, loc='upper right')

# Set x-ticks at each year and rotate labels 45 degrees for better readability
years = reanalysis_om_df['date'].dt.year.unique()
xticks = [reanalysis_om_df[reanalysis_om_df['date'].dt.year == y]['date'].iloc[0] for y in years]
ax1.set_xticks(xticks)
ax1.set_xticklabels(years, rotation=45) #, fontsize=12, color='black')

# Add prominent vertical grid lines at each year to further emphasize years
for xtick in xticks:
    ax1.axvline(x=xtick, color='gray', linestyle='--', linewidth=1, alpha=0.4, zorder=0)

# make horizontal grid lines thinner
ax1.grid(axis='x', which='both', linewidth=0)  # removes default vertical grid lines for ax1
ax1.grid(axis='y', which='both', linewidth=0.5, linestyle=':')
ax2.grid(axis='x', which='both', linewidth=0)  # removes default vertical grid lines for ax2
ax2.grid(axis='y', which='both', linewidth=0.5, linestyle=':')

# ax1.set_xlim(pd.Timestamp('2018-01-01'), pd.Timestamp('2020-12-31'))

plt.title('Temp and Precip Reanalysis Data (Open-Meteo), Bear Valley Visitor Center')
fig.tight_layout()
plt.show()

In [ ]:
# # Plot 7-day moving average of precipitation

# # Calculate 7-day moving average
# reanalysis_om_df['prcp_mm_7d_ma'] = reanalysis_om_df['prcp_mm'].rolling(window=7, min_periods=1).mean()

# fig = go.Figure()

# fig.add_trace(go.Scatter(
#     x=reanalysis_om_df['date'],
#     y=reanalysis_om_df['prcp_mm'],
#     name='Daily Precip',
#     mode='markers',
#     marker_color='green',
#     marker_size=5,
#     opacity=1.0
# ))

# fig.add_trace(go.Scatter(
#     x=reanalysis_om_df['date'],
#     y=reanalysis_om_df['prcp_mm_7d_ma'],
#     name='7-day MA',
#     mode='lines',
#     marker_color='green',
#     marker_size=5,
#     opacity=1.0
# ))

# fig.update_layout(
#     barmode='overlay',
#     title='Open Meteo precip data with 7-day moving average',
#     xaxis_title='Date',
#     yaxis_title='Precipitation (mm)',
#     legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
#     xaxis=dict(
#         rangeselector=dict(
#             buttons=list([
#                 dict(count=1, label="1y", step="year", stepmode="backward"),
#                 dict(count=3, label="3y", step="year", stepmode="backward"),
#                 dict(step="all")
#             ])
#         ),
#         rangeslider=dict(
#             visible=True
#         ),
#         type="date"
#     ),
#     height=400,
#     template='plotly_white'
# )

# fig.show()


# **Mushroom Sightings**

In [ ]:
# Load Mushroom Sightings from iNaturalist
print("Loading iNaturalist data...")
inat_df = pd.read_csv('../Data/inaturalist_data/fungi/observations-675943.csv/observations-675943.csv')

In [ ]:
# --- Define Target Species ---
CHOICE_EDIBLES = {
    # 'Chanterelle': ['Cantharellus californicus', 'Cantharellus formosus'],
    'King Bolete': ['Boletus edulis', 'Boletus edulis var. grandedulis'],
    # 'Candy Cap': ['Lactarius rubidus', 'Lactarius rufulus'],
    # 'Black Trumpet': ['Craterellus cornucopioides', 'Craterellus fallax'],
    # 'Hedgehog Mushroom': ['Hydnum repandum']
}

PROXY_SPECIES = {
    # Chanterelles are found with oaks, just like the highly visible Fly Agaric.
    # 'Chanterelle': [
    #     'Amanita muscaria', # Fly Agaric
    #     'Russula'           # Russula (Genus)
    # ],
    
    # King Boletes are pine-associates, sharing habitat with Suillus and Fly Agaric.
    'King Bolete': [
        'Suillus',          # Slippery Jacks (Genus)
        'Amanita muscaria'  # Fly Agaric
    ],

    # Black Trumpets like damp, mossy areas, similar to colorful Waxcaps and Coral Fungi.
    # 'Black Trumpet': [
    #     'Hygrocybe',        # Waxcaps (Genus)
    #     'Ramaria',          # Coral Fungi (Genus)
    #     'Clavaria'          # Coral Fungi (Genus)
    # ],

    # Candy Caps are a type of milk cap; other common milk caps indicate a suitable habitat.
    # 'Candy Cap': [
    #     'Lactarius alnicola' # A common, non-choice Milk Cap
    # ]
}

In [ ]:
# Filter the inaturalist data using scientific names in CHOICE_EDIBLES and PROXY_SPECIES
# (case insensitive, strip whitespace)
edible_sci_names = set(
    species.strip().lower()
    for species_list in CHOICE_EDIBLES.values()
    for species in species_list
)
proxy_sci_names = set(
    species.strip().lower()
    for species_list in PROXY_SPECIES.values()
    for species in species_list
)
all_target_sci_names = edible_sci_names | proxy_sci_names

edibles_df = inat_df[
    inat_df['scientific_name'].str.strip().str.lower().apply(
        lambda sci: any(target in sci for target in all_target_sci_names)
    )
].copy()

In [ ]:
# Define search area as 2km buffer around Pt Reyes boundary
search_area = pt_reyes_boundary.to_crs(epsg=3857).buffer(2000)

# search_area_web_mercator = search_area.to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 10))
search_area.plot(ax=ax, alpha=1.0, facecolor='none', edgecolor='r', linewidth=2)
cx.add_basemap(ax, source=cx.providers.Esri.WorldTopoMap)
ax.set_axis_off()
plt.show()

In [ ]:
# rf_model for sightings of target species in the rf_model area

# Invert both the CHOICE_EDIBLES and PROXY_SPECIES dictionaries for easy mapping from species name to group
# species_to_group = {}
# for group_dict in [CHOICE_EDIBLES, PROXY_SPECIES]:
#     for group, species_list in group_dict.items():
#         for species in species_list:
#             species_to_group[species] = group

# print(species_to_group)

# Filter for target species and add a 'group' column
# edibles_df = inat_df[inat_df['scientific_name'].isin(species_to_group.keys())].copy()
# edibles_df['group'] = edibles_df['scientific_name'].map(species_to_group)
edibles_df['observed_on'] = pd.to_datetime(edibles_df['observed_on'])

# Add column "proxy_for" to edibles_df, which lists all choice edibles for which the row is a proxy species
EDIBLES_PROXIES_COMBINED = {}
for edible in CHOICE_EDIBLES.keys():
    EDIBLES_PROXIES_COMBINED[edible] = list(CHOICE_EDIBLES.get(edible, []))
    EDIBLES_PROXIES_COMBINED[edible].extend(PROXY_SPECIES.get(edible, []))

# print(EDIBLES_PROXIES_COMBINED)

edibles_df['proxy_for'] = edibles_df['scientific_name'].apply(
    lambda x: [edible for edible in CHOICE_EDIBLES.keys() if any(x.startswith(val) for val in EDIBLES_PROXIES_COMBINED[edible])]
    )

# print(edibles_df[['scientific_name', 'proxy_for']].head())

print(f"Found {len(edibles_df)} choice edible and proxy species sightings.")

# Convert sightings to a GeoDataFrame
edibles_gdf = gpd.GeoDataFrame(
    edibles_df, 
    geometry=gpd.points_from_xy(edibles_df.longitude, edibles_df.latitude),
    crs="EPSG:4326"
)

# Filter sightings to Marin County
search_area_4326 = search_area.to_crs(edibles_gdf.crs)
pt_reyes_edibles_gdf = gpd.clip(edibles_gdf, search_area_4326)

print(f"Found {len(pt_reyes_edibles_gdf)} choice edible and proxy species sightings in Point Reyes region.")

In [ ]:
# Plot all scientific names in the filtered iNaturalist data with at least 20 sightings (most common first)
# import re

order_counts = pt_reyes_edibles_gdf['scientific_name'].value_counts()
order_counts = order_counts[order_counts >= 20]

plot_df = (
    pt_reyes_edibles_gdf[pt_reyes_edibles_gdf['scientific_name'].isin(order_counts.index)]
    .groupby('scientific_name')
    .size()
    .reset_index(name='count')
    .sort_values(by='count', ascending=False)
)

plt.figure(figsize=(12, max(8, 0.5 * len(order_counts))))  # Make figure taller based on number of names

ax = sns.barplot(
    data=plot_df,
    y='scientific_name',
    x='count',
    order=plot_df['scientific_name'],  # ensures identical order
    color='skyblue',
    linewidth=0,
    orient='h',
)

plt.title('Sightings by Scientific Name (Choice Edibles in Bold)')
plt.xlabel('Number of Sightings')

# --- Make y-axis label bold for any choice edible scientific names ---
# Build a flat set of all edible scientific names (lowercased & stripped) from the CHOICE_EDIBLES dict
edibles_sci_names = set(
    species.strip().lower()
    for species_list in CHOICE_EDIBLES.values()
    for species in species_list
)
# Get all tick labels, compare (case-insensitive) if ANY edible name is a substring, and set fontweight
for label in ax.get_yticklabels():
    sci_name = label.get_text().strip().lower()
    if any(edible in sci_name for edible in edibles_sci_names):
        label.set_fontweight('bold')

plt.ylabel('Scientific Name')
plt.tight_layout()
plt.show()

## Sighting Map of Point Reyes region

In [ ]:
# Fix: Preserve original point geometries
fig, ax = plt.subplots(figsize=(15, 15))

# Plot Marin boundary for context
pt_reyes_boundary_web = pt_reyes_boundary.to_crs(epsg=3857)
pt_reyes_boundary_web.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1)

# Use the original pt_reyes_edibles_gdf (which has Point geometries) instead of enriched_gdf
plot_gdf = pt_reyes_edibles_gdf.to_crs(epsg=3857)  # Use original points

# Only keep rows where geometry is Point
points_only = plot_gdf[plot_gdf.geometry.geom_type == "Point"].copy()


# Extract x/y for scatterplot
points_only["x"] = points_only.geometry.apply(lambda geom: geom.x)
points_only["y"] = points_only.geometry.apply(lambda geom: geom.y)

# Get unique groups and colors
edibles = CHOICE_EDIBLES.keys()
colors = sns.color_palette("Set2", n_colors=len(edibles))

# Plot each choice edible group separately with edge colors
for i, edible in enumerate(edibles):
    # edible_data = points_only[points_only["proxy_for"].apply(lambda proxies: isinstance(proxies, list) and "edible" in proxies)]
    edible_data = points_only[points_only["proxy_for"].apply(lambda proxy_edibles: edible in proxy_edibles)]
    # print(edible_data.head())
    
    # Create darker edge color
    edge_color = tuple([max(c * 0.6, 0) for c in colors[i]])
    
    ax.scatter(
        edible_data["x"], 
        edible_data["y"], 
        c=[colors[i]], 
        s=50, 
        alpha=0.8,
        linewidth=0.8,
        edgecolor=edge_color,
        label=edible
    )

# Add basemap
cx.add_basemap(ax, source=cx.providers.Esri.WorldTopoMap, zoom=13)

ax.set_title('Porcini/Proxy Sightings in Point Reyes', fontsize=16, fontweight='bold')
ax.set_axis_off()
plt.legend(title='Species Group')
plt.show()

print(f"Plotted {len(points_only)} mushroom sightings")

## Sightings over time

In [ ]:
print(pt_reyes_edibles_gdf.columns)
pt_reyes_edibles_gdf.head()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots



# Count reports by date
reports_by_date = pt_reyes_edibles_gdf.groupby('observed_on').size().reset_index(name='count')
# Our weather data starts in 2010 - remove any sightings before then (**iNaturalist was create in 2008, first app was launched in 2011**)
reports_by_date = reports_by_date[reports_by_date['observed_on'] >= pd.Timestamp('2010-01-01')]
print(reports_by_date.head())

# Ensure date columns are parsed as datetime
reports_by_date['observed_on'] = pd.to_datetime(reports_by_date['observed_on'])

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])


# Plot Precipitation (mm) - Open-Meteo
fig.add_trace(go.Scatter(
    x=reanalysis_om_df['date'],
    y=reanalysis_om_df['prcp_mm'],
    name='Daily Precip',
    mode='markers',
    marker_color='green',
    marker_size=1,
    opacity=1.0
))

# Plot Precipitation (mm) - Open-Meteo 7-day moving average
fig.add_trace(go.Scatter(
    x=reanalysis_om_df['date'],
    y=reanalysis_om_df['prcp_mm_7d_ma'],
    name='Precip 7-day MA',
    mode='lines',
    marker_color='green',
    line_width=1,
    opacity=1.0
))

# Plot mushroom observation counts on a secondary y-axis (right)
fig.add_trace(go.Bar(
    x=reports_by_date['observed_on'],
    y=reports_by_date['count'],
    name='Mushroom Sightings',
    marker_color='brown',
    opacity=1.0
    ),
    secondary_y=True
)


fig.update_layout(
    barmode='overlay',
    title='Mushroom Sightings by Date, with Precipitation',
    xaxis_title='Date',
    yaxis_title='Number of Sightings',
    legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=3, label="3y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ),
    height=400,
    template='plotly_white'
)

fig.show()

## Engineer Features

### Merge OpenMeteo and iNaturalist datasets

In [ ]:
model_data = reports_by_date.merge(reanalysis_om_df, left_on='observed_on', right_on='date', how='outer')
model_data['count'] = model_data['count'].fillna(0)
model_data.head(20)


In [ ]:
# 14-day precipitation (simple moving sum)
model_data['14_day_prcp_mm'] = model_data['prcp_mm'].rolling(window=14, closed='left').sum()
# 30-day precipitation (simple moving sum)
model_data['30_day_prcp_mm'] = model_data['prcp_mm'].rolling(window=30, closed='left').sum()
# 60-day precipitation (simple moving sum)
model_data['60_day_prcp_mm'] = model_data['prcp_mm'].rolling(window=60, closed='left').sum()

# Exponential moving averages with half-lives of 7, 14, 30, and 60 days
model_data['prcp_mm_ema_hl7'] = model_data['prcp_mm'].ewm(halflife=7, adjust=False).mean()
model_data['prcp_mm_ema_hl14'] = model_data['prcp_mm'].ewm(halflife=14, adjust=False).mean()
model_data['prcp_mm_ema_hl30'] = model_data['prcp_mm'].ewm(halflife=30, adjust=False).mean()
model_data['prcp_mm_ema_hl60'] = model_data['prcp_mm'].ewm(halflife=60, adjust=False).mean()

# model_data['14_day_prcp_mm'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
# plt.xlabel('14-day Precipitation (mm)')
# plt.show()


In [ ]:
# # Compute days since last precip >= 1mm for each day
# last_precip_idx = None
# days_since_last_precip = []
# for idx, row in model_data.iterrows():
#     # Counts days since precip >= 1mm 
#     # ***before the current day - we want to avoid effect of poor observer effort on rainy days
#     if last_precip_idx is None:
#         days_since_last_precip.append(None)
#     else:
#         days_since_last_precip.append(idx - last_precip_idx)

#     if row['prcp_mm'] >= 1:
#         last_precip_idx = idx
        

# model_data['days_since_last_precip_over_1mm'] = days_since_last_precip

# # plt.plot(model_data['days_since_last_precip_over_1mm'], model_data['count'], 'o', markersize=1)
# # plt.xlabel('Days Since Last Precip >= 1mm')
# # plt.ylabel('Sighting Count')
# # # plt.xlim(0,10)
# # plt.show()

# model_data['days_since_last_precip_over_1mm'].plot.hist(bins=40, weights=model_data['count']) #, edgecolor='black')
# plt.xlabel('Days Since Last Precip >= 1mm')
# plt.show()


In [ ]:
model_data['7day_tmax_c'] = model_data['tmax_c'].rolling(window=7, closed='left').mean()
# model_data['7day_tmax_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
# plt.xlabel('7-day Moving Average of Max Temp (C)')
# plt.show()

model_data['7day_tmin_c'] = model_data['tmin_c'].rolling(window=7, closed='left').mean()
# model_data['7day_tmin_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
# plt.xlabel('7-day Moving Average of Min Temp (C)')
# plt.show()



In [ ]:
model_data['14day_tmax_c'] = model_data['tmax_c'].rolling(window=14, closed='left').mean()
# model_data['14day_tmax_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
# plt.xlabel('14-day Moving Average of Max Temp (C)')
# plt.show()

model_data['14day_tmin_c'] = model_data['tmin_c'].rolling(window=14, closed='left').mean()
# model_data['14day_tmin_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
# plt.xlabel('14-day Moving Average of Min Temp (C)')
# plt.show()

model_data['14-7_tmax_c'] = model_data['14day_tmax_c']-model_data['7day_tmax_c']
# model_data['14-7_tmax_c'].plot.hist(bins=10, weights=model_data['count']) #, edgecolor='black')
# plt.xlabel('14-day Moving Average of Max Temp (C) - 7-day Moving Average of Max Temp (C)')
# plt.show()

model_data['14-7_tmin_c'] = model_data['14day_tmin_c']-model_data['7day_tmin_c']
# model_data['14-7_tmin_c'].plot.hist(bins=10, weights=model_data['count']) #, edgecolor='black')
# plt.xlabel('14-day Moving Average of Min Temp (C) - 7-day Moving Average of Min Temp (C)')
# plt.show()



In [ ]:
model_data['dayofweek'] = model_data['date'].dt.dayofweek

model_data['is_weekend'] = model_data['dayofweek'].isin([5,6])
model_data['is_weekend'].value_counts()

In [ ]:
# Include the day of year in sin/cos time
model_data['sin_time'] = np.sin(2 * np.pi * model_data['yday'] / 365)
model_data['cos_time'] = np.cos(2 * np.pi * model_data['yday'] / 365)

# plt.plot(model_data['yday'], model_data['sin_time'], 'r-', label='sin_time')
# plt.plot(model_data['yday'], model_data['cos_time'], 'b-', label='cos_time')
# plt.legend()
# plt.show()


In [ ]:
print(model_data.tail())

In [ ]:
# Filter to 2016-2026 (more iNaturalist data, end on an even year for time series split)
start_year = 2016
end_year = 2026

start_date = f'{start_year}-01-01'
end_date = f'{end_year-1}-12-31'

model_data_filtered = model_data[(model_data['date'] >= start_date) & (model_data['date'] <= end_date)].copy()

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Define a time series split for all cross-validation
test_years = 2 # Leave a couple years for testing
cv = TimeSeriesSplit(n_splits=(end_year-start_year-test_years))

# Mark the training and testing period
split_idx = int(len(model_data_filtered) * (1 - test_years/(end_year-start_year)))
model_data_filtered['train_test'] = 'train'
model_data_filtered.iloc[split_idx:, model_data_filtered.columns.get_loc('train_test')] = 'test'

In [ ]:
model_data_filtered.head()

## Latest modelling approach

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

model_features = [
    # observer effort features
    'tmax_c', 'tmin_c', 'prcp_mm', 'is_weekend', #'dayofweek',
    # precip features
    # '14_day_prcp_mm', '30_day_prcp_mm', '60_day_prcp_mm',     
    'prcp_mm_ema_hl7','prcp_mm_ema_hl14','prcp_mm_ema_hl30','prcp_mm_ema_hl60',
    # 'days_since_last_precip_over_1mm', 'days_since_last_precip_over_3mm', 
    # 'days_since_last_precip_over_5mm',
    # temperature features
    '7day_tmax_c', '7day_tmin_c', '14day_tmax_c', '14day_tmin_c', '14-7_tmax_c', '14-7_tmin_c',
    # time features (exclude - mushrooms don't know the date)
    'sin_time', 'cos_time'
]

X = model_data_filtered[model_features].copy()
# X['ones'] = 1
y = model_data_filtered['count'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=(test_years/(end_year-start_year)), shuffle=False)

In [ ]:
# print(max(model_data_filtered['count']))
# print(max(y_train))
# print(max(y_test))
# print(test_years/(end_year-start_year))
# print(len(y_test))

#### Random Forest model

In [ ]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import KFold

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 20, 160, step=20),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'min_samples_split': trial.suggest_int('min_samples_split', 8, 16),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 4, 16),
        'max_features': trial.suggest_int('max_features', 1, 10),
        'n_jobs': -1,
        'random_state': 42
    }

    rf_model = RandomForestRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')
    return -scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

best_params = study.best_params
print(f"Best parameters found: {best_params}")

# Train final model with the best found hyperparameters
best_rf = RandomForestRegressor(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    n_jobs=-1,
    random_state=42
)

best_rf.fit(X_train, y_train)

y_pred = best_rf.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse}")
print(f"R2: {r2}")

plt.scatter(y_test, y_pred)
plt.xlabel('Actual Count')
plt.ylabel('Predicted Count')
plt.title('Actual vs Predicted Counts')
plt.show()

In [ ]:
model_data_filtered["pred_count_rf"] = best_rf.predict(X)

In [ ]:
columns = X.columns
# Get the best RandomForestRegressor from the pipeline
# best_rf = rf_model.best_estimator_.named_steps['model']
print(best_rf.feature_importances_)

# Plot the feature importances
plt.bar(columns, best_rf.feature_importances_)
plt.xlabel('Features')
plt.ylabel('Importance')
plt.xticks(rotation=90)
plt.show()

In [ ]:
print(y)

#### XGBoost model

In [ ]:
import optuna
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "n_estimators": 1500,
        "early_stopping_rounds": 30,
        "random_state": 42
    }

    X_train_opt, X_valid_opt = X_train, X_test   # Early stopping for demo: use test here, or better: use validation split from X_train
    y_train_opt, y_valid_opt = y_train, y_test

    model = XGBRegressor(**params)
    # Fit with early stopping
    model.fit(
        X_train_opt, y_train_opt,
        eval_set=[(X_valid_opt, y_valid_opt)],
        # early_stopping_rounds=10,
        verbose=False
    )
    preds = model.predict(X_valid_opt)
    mse = mean_squared_error(y_valid_opt, preds)
    return mse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best parameters for XGBoost:", study.best_params)

best_xgb_model = XGBRegressor(
    **study.best_params,
    random_state=42,
    objective='reg:squarederror',
    eval_metric='rmse'
)
best_xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    # early_stopping_rounds=10,
    verbose=False
)

y_pred_xgb = best_xgb_model.predict(X_test)

mse_xgb = mean_squared_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"XGBoost MSE: {mse_xgb}")
print(f"XGBoost R2: {r2_xgb}")

plt.scatter(y_test, y_pred_xgb)
plt.xlabel('Actual Count')
plt.ylabel('Predicted XGBoost Count')
plt.title('Actual vs Predicted Counts (XGBoost)')
plt.show()

In [ ]:
import xgboost as xgb

xgb.plot_importance(best_xgb_model, importance_type='gain', height=0.4)
plt.title('Feature Importance (Gain)')
plt.show()

## Model evaluation

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])


# Plot Precipitation (mm) - Open-Meteo
fig.add_trace(go.Bar(
    x=model_data_filtered['date'],
    y=model_data_filtered['prcp_mm'],
    name='Precipitation (mm) - Open-Meteo',
    # mode='markers',
    # marker_color='blue',
    # marker_size=1,
    opacity=1.0
))

# # Plot Precipitation (mm) - Open-Meteo 7-day moving average
# fig.add_trace(go.Scatter(
#     x=model_data_filtered['date'],
#     y=model_data_filtered['prcp_mm_7d_ma'],
#     name='Precipitation (mm) - 7-day moving average',
#     mode='lines',
#     marker_color='green',
#     line_width=1,
#     opacity=1.0
# ))

# Plot mushroom observation counts on a secondary y-axis (right)
model_data_filtered = model_data_filtered.sort_values(by='observed_on')
fig.add_trace(go.Scatter(
    x=model_data_filtered['date'],
    y=model_data_filtered['count'],
    name='Mushroom Sightings',
    mode='markers',
    marker_color='brown',
    marker_size=5,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

# # Plot Precipitation (mm) - Open-Meteo 7-day moving average
# fig.add_trace(go.Scatter(
#     x=reanalysis_om_df['date'],
#     y=reanalysis_om_df['prcp_mm_7d_ma'],
#     name='Precipitation (mm) - 7-day moving average',
#     mode='lines',
#     marker_color='green',
#     line_width=1,
#     opacity=1.0
# ))

# Plot predicted mushroom observation counts from RF model
predX_rf = model_data_filtered.copy()
predX_rf['predicted_count'] = best_rf.predict(predX_rf[model_features])
predX_rf = predX_rf.sort_values(by='date')
fig.add_trace(go.Scatter(
    x=predX_rf['date'],
    y=predX_rf['predicted_count'],
    name='Predicted Sightings',
    mode='lines+markers',
    marker_color='green',
    marker_size=4,
    line_width=1,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

# Plot predicted mushroom observation counts from XGBoost model
predX_xgb = model_data_filtered.copy()
predX_xgb['predicted_count'] = best_xgb_model.predict(predX_xgb[model_features])
predX_xgb = predX_xgb.sort_values(by='date')
fig.add_trace(go.Scatter(
    x=predX_xgb['date'],
    y=predX_xgb['predicted_count'],
    name='Predicted Sightings',
    mode='lines+markers',
    marker_color='red',
    marker_size=4,
    line_width=1,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

# print(predX['observed_on'].min())

# Plot predicted mushroom observation counts under ideal conditions (weekend, good weather)
# predX_ideal = predX.copy()
# predX_ideal['is_weekend'] = True
# predX_ideal['tmax_c'] = 12
# predX_ideal['tmin_c'] = 8
# predX_ideal['prcp_mm'] = 0
# predX_ideal['predicted_count'] = rf_model.predict(predX_ideal[model_features])
# predX_ideal = predX_ideal.sort_values(by='date')
# fig.add_trace(go.Scatter(
#     x=predX_ideal['date'],
#     y=predX_ideal['predicted_count'],
#     name='Predicted Sightings (Ideal)',
#     mode='lines+markers',
#     marker_color='blue',
#     marker_size=4,
#     line_width=1,
#     marker_symbol='circle',
#     opacity=1.0
#     ),
#     secondary_y=True
# )

fig.update_layout(
    barmode='overlay',
    title='Mushroom Sightings and Predictions by Date, with Precipitation',
    xaxis_title='Date',
    yaxis_title='Number of Sightings',
    legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=3, label="3y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ),
    height=400,
    template='plotly_white'
)

fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(predX_rf['cos_time'], predX_rf['predicted_count'], linewidth=1)
ax.plot(predX_xgb['cos_time'], predX_xgb['predicted_count'], linewidth=1, color='red')
# ax.plot([366-182, 366-182], [0, 8], 'k--', linewidth=1, label='Jan 1st')
ax.set_xlabel('Cosine Time')
ax.set_ylabel('Predicted Count')
ax.set_title('Predicted Count by Cosine Time')
ax.legend()
plt.show()


In [ ]:
# Save the best model
import pickle

# Save the best model to a file
import datetime
current_date = datetime.datetime.now().strftime('%Y-%m-%d')
with open(f'model_{current_date}.pkl', 'wb') as f:
    pickle.dump(rf_model.best_estimator_, f)


In [ ]:
predX[['date'] + model_features].to_csv('predX.csv', index=False)